# Day 2-04｜YOLO Detection、Court Keypoint 與 BEV 影片整合
> Python 籃球運動資料分析課程  
> 本單元整合 detector、court keypoint model、Homography 與 BEV 投影，對實際影片輸出左右並排的分析影片。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 使用 YOLO detector 取得 player bbox。
- 使用 court keypoint model 估計 frame-to-BEV Homography。
- 將 player footpoint 投影到 BEV 並輸出整合影片。


## 課程流程
1. 選擇另一支參考影片。
2. 對每個 frame 執行 detection 與 court keypoint。
3. 從 court keypoint 較穩定的 frame 開始，產生 `d2_04_detector_keypoint_bev.mp4` 與對應 JSON。


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("找不到 src/course_setup.py，請確認目前目錄位於課程 repo 內。")

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT)


In [ ]:
from src.video_utils import display_video_in_notebook
from src.yolo_utils import (
    court_keypoint_model_path,
    detector_model_path,
    reference_videos,
    write_detector_keypoint_bev_video,
)

videos = reference_videos(COURSE_ROOT)
if len(videos) < 2:
    raise FileNotFoundError("assets/raw/reference_videos/ 至少需要兩支參考影片。")

VIDEO_PATH = videos[1]
DETECTOR_PATH = detector_model_path(COURSE_ROOT)
COURT_MODEL_PATH = court_keypoint_model_path(COURSE_ROOT)
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"

print("video:", VIDEO_PATH)
print("detector:", DETECTOR_PATH)
print("court model:", COURT_MODEL_PATH)
print("start frame:", 90)


In [ ]:
bev_video, rows = write_detector_keypoint_bev_video(
    video_path=VIDEO_PATH,
    detector_path=DETECTOR_PATH,
    court_model_path=COURT_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=COURSE_ROOT / "assets" / "results" / "d2_04_detector_keypoint_bev.mp4",
    max_frames=90,
    detector_conf=0.25,
    keypoint_conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=90,
)

print("BEV video:", bev_video)
print("BEV json:", bev_video.with_suffix(".json"))
print("projected rows:", len(rows))
display_video_in_notebook(bev_video, loop=True)


本單元輸出的 BEV 點位尚未維持跨 frame 身分；Day 3 會加入 ByteTrack，讓同一位球員在不同 frame 中保留穩定 track ID。